# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [3]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
orders_df = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)
# TODO: assert the row count and the quantity total
assert len(orders_df) == 4, f"Expected 4 rows, got {len(orders_df)}"
assert orders_df['qty'].sum() == 7, f"Expected total quantity of 7, got {orders_df['qty'].sum()}"

print("Q1 passed!")
orders_df

Q1 passed!


,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [4]:
vendors_df = pd.json_normalize(vendors_json)

joined_df = pd.merge(
    orders_df,
    vendors_df,
    on='vendor_id',
    how='left',
    indicator=True
)

unmatched = joined_df[joined_df['_merge'] == 'left_only']

if not unmatched.empty:
    unmatched_vendors = unmatched['vendor_id'].unique()
    unmatched_units = unmatched['qty'].sum()
    print(f"Unmatched vendor(s): {unmatched_vendors}, total units attached: {unmatched_units}")
else:
    print("All vendors matched in the provided payload.")

joined_df

All vendors matched in the provided payload.


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [5]:
qty_per_vendor = (
    joined_df.assign(name=joined_df['name'].fillna('Unmatched / Unknown Vendor'))
    .groupby('name')['qty']
    .sum()
    .reset_index()
)

qty_per_vendor

,name,qty
0,Cav Merch,1
1,Hoos Burgers,6


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [6]:
zone_totals = joined_df.groupby('zone', dropna=False)['qty'].sum()

print("Quantity sold per zone (including NaN zones):")
print(zone_totals)

matched_total = joined_df.dropna(subset=['zone'])['qty'].sum()
total_qty = joined_df['qty'].sum()
diff = total_qty - matched_total

print(f"\nTotal quantity: {total_qty}")
print(f"Matched zone quantity: {matched_total}")
print(f"Difference: {diff} unit(s) (attributable to unmatched vendors with no assigned zone).")

Quantity sold per zone (including NaN zones):
zone
A    7
Name: qty, dtype: int64

Total quantity: 7
Matched zone quantity: 7
Difference: 0 unit(s) (attributable to unmatched vendors with no assigned zone).


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [10]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

safe_ragged_json = [
    {**order, 'lines': order.get('lines', [])}
    for order in ragged_json
]

ragged_df = pd.json_normalize(
    safe_ragged_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

if 'qty' in ragged_df.columns:
    ragged_df['qty'] = ragged_df['qty'].astype('Int64')

ragged_df

,item,qty,order,vendor_id
0,Soda,2,4,V-01
1,Fries,<NA>,6,V-01


### Q6 — Validate

In [11]:
# assert len(lines) == 4
# assert lines['qty'].sum() == 7
# assert len(joined) == len(lines), 'the vendor join changed the row count'
assert len(orders_df) == 4
assert orders_df['qty'].sum() == 7
assert len(joined_df) == len(orders_df), 'the vendor join changed the row count'
print('checks passed.')
print('checks passed.')

checks passed.
checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a. One row per line item represents the fundamental granular transaction unit. This shape allows direct aggregation on item quantities, pricing, or product categories. If structured with one row per order, line items would remain trapped in nested lists or wide arrays, making operations like calculating total sales per item or joining item-level attributes more complex.

b. A missing quantity represents unrecorded or omitted data, whereas a quantity of zero explicitly represents a recorded event where no items were purchased. Filling missing values with zero using fillna(0) artificially inflates zero-count stats, distorts summary statistics, and prevents downstream validation pipelines from detecting data ingestion failures.